# SPARCS aggregation (sparcs_aggregation.ipynb)

**Purpose:** Aggregate SPARCS asthma hospitalization records (by zip3) and map to boroughs to produce borough-level summaries for analysis.

**Input:** data_csv/SPARCS_asthma_hospitalizations_by_zip3_2010-2024.csv

**Dependencies:** pandas

**How to run:** Run the notebook to generate aggregated tables; ensure the SPARCS CSV is present in `data_csv/`.


In [1]:
# Consolidated imports
import pandas as pd

In [2]:
# Load your SPARCS CSV
df = pd.read_csv("data_csv/SPARCS_asthma_hospitalizations_by_zip3_2010-2024.csv")

zip3_to_borough = {
    # NYC - Manhattan (New York County)
    100: "Manhattan",
    101: "Manhattan",

    # NYC - Bronx (Bronx County)
    104: "Bronx",

    # NYC - Staten Island (Richmond County)
    103: "Staten Island",

    # NYC - Queens (Queens County)
    110: "Queens",
    111: "Queens",
    113: "Queens",
    114: "Queens",
    116: "Queens",

    # NYC - Brooklyn (Kings County)
    112: "Brooklyn",

    # NYC - Other / Mixed NYC (used by USPS for PO Boxes, unique zips, etc.)
    102: "Manhattan",   # PO Boxes / unique Manhattan zips
    105: "Bronx",       # Bronx overlap
    106: "Bronx",       # Bronx overlap

    # New York State - Outside NYC
    107: "Westchester County",
    108: "Westchester County",
    109: "Westchester County",
    115: "Nassau County",
    117: "Nassau County",
    118: "Suffolk County",
    119: "Suffolk County",

    # Hudson Valley / Capital Region
    120: "Albany Area",
    121: "Albany Area",
    122: "Albany Area",
    123: "Schenectady/Albany Area",
    124: "Hudson Valley",
    125: "Hudson Valley",
    126: "Hudson Valley",
    127: "Hudson Valley",
    128: "Catskills",
    129: "Hudson Valley",

    # Central/Western NY
    130: "Syracuse Area",
    131: "Syracuse Area",
    132: "Syracuse Area",
    133: "Utica Area",
    134: "Utica Area",
    135: "Utica Area",
    136: "Watertown Area",
    137: "Binghamton Area",
    138: "Binghamton Area",
    139: "Binghamton Area",

    # Western NY
    140: "Buffalo Area",
    141: "Buffalo Area",
    142: "Buffalo Area",
    143: "Niagara Falls Area",
    144: "Rochester Area",
    145: "Rochester Area",
    146: "Rochester Area",
    147: "Corning/Elmira Area",
    148: "Ithaca/Elmira Area",
    149: "Elmira Area",
}

# filter to keep only NYC's five boroughs
nyc_zip3s = {100, 101, 102, 103, 104, 105, 106, 110, 111, 112, 113, 114, 116}

df = df[df['zip3'].isin(nyc_zip3s)]

nyc_boroughs = {"Manhattan", "Bronx", "Brooklyn", "Queens", "Staten Island"}

# Map borough
df['Borough'] = df['zip3'].map(zip3_to_borough)
df = df[df['Borough'].isin(nyc_boroughs)]

# Check if any zip3 wasn't mapped
unmapped = df[df['Borough'].isna()]['zip3'].unique()
if len(unmapped) > 0:
    print(f"Unmapped zip3s: {unmapped}")

# Aggregate by Year and Borough
agg_funcs = {
    'Asthma_Cases': 'sum',
    'ED_Visits': 'sum',
    'Avg_Length_of_Stay': 'mean',
    'Avg_Severity': 'mean',
    'Avg_Mortality_Risk': 'mean',
    'Total_Charges': 'sum',
    'Total_Costs': 'sum',
    'Avg_Costs': 'mean',
    'Medicaid_Cases': 'sum',
    'ED_Rate': 'mean',
    'Medicaid_Rate': 'mean'
}

df_agg = df.groupby(['Year', 'Borough']).agg(agg_funcs).reset_index()

# Optional: round numeric columns for readability
numeric_cols = df_agg.select_dtypes('number').columns
df_agg[numeric_cols] = df_agg[numeric_cols].round(2)

df_agg.to_csv("data_csv/SPARCS_by_borough_2010-2024.csv", index=False)